# 🗺️ Game Terrain Generator — PromptFlow

### Pipeline
```
                    GPU: 16GB T4
┌──────────────────────────────────────────────┐
│  Phase 1 (LLM thinking)                      │
│  Llama-3.2-3B-Instruct-uncensored  ~2GB GPU  │
│    └─ Analyze game context                   │
│    └─ Generate per-tile image prompts         │
│                                              │
│  ⬇  offload LLM to CPU (frees ~2GB VRAM)    │
│                                              │
│  Phase 2 (Image gen)                         │
│  Z-Image Turbo GGUF  ~10GB GPU              │
│    └─ Generate all terrain tiles             │
└──────────────────────────────────────────────┘
```

**Models used:**
- `bartowski/Llama-3.2-3B-Instruct-uncensored-GGUF` — uncensored local LLM Q4_K_M (~2GB)
- `unsloth/Z-Image-Turbo-GGUF` — fast image generation GGUF
- `unsloth/Qwen3-4B-Instruct-2507-GGUF` — LLM component inside Z-Image
- `black-forest-labs/FLUX.1-schnell` — VAE component

In [ ]:
# CELL 1: Install dependencies
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!pip install stable-diffusion-cpp-python huggingface_hub rich pyyaml requests pillow openai -q
print('✅ Dependencies installed')

In [ ]:
# CELL 2: Download all model files
import os
from huggingface_hub import hf_hub_download

# Uncensored local LLM (bartowski, abliterated Llama 3.2 3B)
LOCAL_LLM_REPO = 'bartowski/Llama-3.2-3B-Instruct-uncensored-GGUF'
LOCAL_LLM_FILE = 'Llama-3.2-3B-Instruct-uncensored-Q4_K_M.gguf'

# Z-Image Turbo components
DIFF_REPO     = 'unsloth/Z-Image-Turbo-GGUF'
DIFF_FILE     = 'z-image-turbo-Q4_K_M.gguf'
LLM_DIFF_REPO = 'unsloth/Qwen3-4B-Instruct-2507-GGUF'
LLM_DIFF_FILE = 'Qwen3-4B-Instruct-2507-Q4_K_M.gguf'
VAE_REPO      = 'black-forest-labs/FLUX.1-schnell'
VAE_FILE      = 'ae.safetensors'

print('Downloading models...')
local_llm_path = hf_hub_download(repo_id=LOCAL_LLM_REPO, filename=LOCAL_LLM_FILE)
diff_path      = hf_hub_download(repo_id=DIFF_REPO,      filename=DIFF_FILE)
llm_diff_path  = hf_hub_download(repo_id=LLM_DIFF_REPO,  filename=LLM_DIFF_FILE)
vae_path       = hf_hub_download(repo_id=VAE_REPO,        filename=VAE_FILE)

print(f'\n✅ Local uncensored LLM : {local_llm_path}')
print(f'✅ Z-Image diffusion    : {diff_path}')
print(f'✅ Z-Image LLM component: {llm_diff_path}')
print(f'✅ VAE                  : {vae_path}')

In [ ]:
# CELL 3: Start model_server.py
# Loads local uncensored LLM on GPU first, then Z-Image Turbo.
# Exposes /llm/chat, /llm/offload, /llm/reload, /image/txt2img, /image/img2img, /health
import sys, subprocess, time, requests

SERVER_PORT = 7860
SERVER_URL  = f'http://localhost:{SERVER_PORT}'

server_proc = subprocess.Popen(
    [
        sys.executable, 'model_server.py',
        '--llm_path',      local_llm_path,
        '--diff_path',     diff_path,
        '--llm_diff_path', llm_diff_path,
        '--vae_path',      vae_path,
        '--llm_gpu_layers', '99',
        '--port', str(SERVER_PORT),
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

print('⏳ Starting model_server.py...')
for line in server_proc.stdout:
    print(line, end='')
    if 'listening at' in line.lower():
        break

print('\n⏳ Waiting for models to load...')
for attempt in range(120):
    try:
        h = requests.get(f'{SERVER_URL}/health', timeout=3).json()
        llm_ok = h['llm']['loaded']
        img_ok = h['image']['loaded']
        free   = h['gpu_free_mb']
        print(f'  LLM={"✅" if llm_ok else "⏳"}  Image={"✅" if img_ok else "⏳"}  GPU free={free}MB', end='\r')
        if llm_ok and img_ok:
            print(f'\n\n✅ model_server ready! GPU free: {free} MB')
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print('\n⚠️  Still loading — continuing anyway.')

In [ ]:
# CELL 4: Quick test — local uncensored LLM
import requests

resp = requests.post(f'{SERVER_URL}/llm/chat', json={
    'messages': [{'role': 'user', 'content': 'Name 3 dramatic terrain types for a dark fantasy RTS game. One sentence each.'}],
    'temperature': 0.9,
    'max_tokens': 200,
}).json()

print('🤖 Local uncensored LLM test:')
print('─' * 50)
print(resp.get('content', resp))
print('─' * 50)

h = requests.get(f'{SERVER_URL}/health').json()
print(f'\nGPU free={h["gpu_free_mb"]}MB  LLM on_gpu={h["llm"]["on_gpu"]}')

In [ ]:
# CELL 5: Configure and run the PromptFlow
# The .promptflow config uses llm.provider=local and gpu_lifecycle.offload_before_image_gen=true
# The interpreter will automatically POST /llm/offload before image generation starts.
import sys, os
sys.path.insert(0, 'interpreter')
from interpreter.interpreter import PromptFlowInterpreter

# ── CONFIGURE YOUR GAME ───────────────────────────────────────────
GAME_CONTEXT = """
A photorealistic RTS set during the late Cretaceous. Players control dinosaur factions
across lush jungles, volcanic plains, coastal marshlands, and highland cliffs.
Art style: photorealistic CGI, cinematic grade. Each terrain tile is a floating
isometric platform with the dirt cross-section visible on all sides. Black background.
"""

TERRAIN_INSTRUCTIONS = """
Include: flat grass, jungle floor with roots, volcanic rock, coastal marsh, cliff edges
(all 4 sides + 4 corners), river crossing, volcanic crater. Edge-connectable tiles.
Consistent lighting: overcast midday from upper-left.
"""

TERRAIN_ASSETS_TO_MAKE = 25
OUTPUT_DIR = 'output/terrain_tiles'
SAMPLE_IMAGE = 'sample_terrain.png'
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
# ─────────────────────────────────────────────────────────────────

pf = PromptFlowInterpreter(
    promptflow_path='promptflows/game_terrain_generator.promptflow',
    cli_inputs={
        'game_context':               GAME_CONTEXT,
        'terrain_asset_instructions': TERRAIN_INSTRUCTIONS,
        'terrain_assets_to_make':     TERRAIN_ASSETS_TO_MAKE,
        'out_dir':                    OUTPUT_DIR,
    },
    openai_api_key=OPENAI_API_KEY,
    model_server_url=SERVER_URL,
    sample_image_path=SAMPLE_IMAGE if os.path.exists(SAMPLE_IMAGE) else None,
    dry_run=False,
)

results = pf.run()

In [ ]:
# CELL 6: Gallery view
from pathlib import Path
from IPython.display import display, HTML

images = sorted(Path(OUTPUT_DIR).glob('terrain_*.png'))
print(f'🖼️  {len(images)} terrain tiles in {OUTPUT_DIR}')

html = "<div style='display:flex;flex-wrap:wrap;gap:10px;padding:8px;background:#111'>"
for p in images:
    html += f"<div style='text-align:center'><img src='{p}' width='200' style='border:1px solid #444;border-radius:3px'><br><span style='font-size:10px;color:#888;font-family:monospace'>{p.stem}</span></div>"
html += '</div>'
display(HTML(html))

In [ ]:
# CELL 7: GPU / model status monitor
import requests, json
h = requests.get(f'{SERVER_URL}/health').json()
print(json.dumps(h, indent=2))
# Manual controls:
# requests.post(f'{SERVER_URL}/llm/offload')  # free VRAM
# requests.post(f'{SERVER_URL}/llm/reload')   # restore LLM to GPU

In [ ]:
# CELL 8: Dry run — see the generated prompts without spending GPU time
import sys; sys.path.insert(0, 'interpreter')
from interpreter.interpreter import PromptFlowInterpreter
import os

pf_dry = PromptFlowInterpreter(
    promptflow_path='promptflows/game_terrain_generator.promptflow',
    cli_inputs={
        'game_context':               GAME_CONTEXT,
        'terrain_asset_instructions': TERRAIN_INSTRUCTIONS,
        'terrain_assets_to_make':     10,
        'out_dir':                    'output/dry_run',
    },
    openai_api_key=os.environ.get('OPENAI_API_KEY', ''),
    model_server_url=SERVER_URL,
    dry_run=True,
)

dry = pf_dry.run()
manifest = dry.get('generate_asset_list', {}).get('asset_manifest', [])
if manifest:
    print(f'\n📋 {len(manifest)} tiles:')
    for a in manifest:
        print(f'  [{a["id"]}] {a["category"]} — {a["name"]}')
        print(f'    {a["image_prompt"][:120]}...')
        print()

In [ ]:
# CELL 9: Stop server
try:
    server_proc.terminate()
    print('✅ model_server stopped.')
except Exception as e:
    print(f'Stop: {e}')